# Heart Disease Risk - Explained Simply

## What are we doing?

A doctor measures easy things about a patient: age, blood pressure, cholesterol, peak heart
rate during a stress test, and so on. The doctor wants to guess one thing: **does this person
have a serious blocked artery in their heart?**

The only way to know for sure is an invasive test, which is expensive and a little risky.
So we build a cheap first guess. If our guess says "probably blocked," the doctor knows to
send that patient for the real test first.

We are **not** diagnosing anyone. We are building a **risk calculator**. Think of a weather
app that says "80% chance of rain" - it does not make it rain, it just helps you decide
whether to grab an umbrella.

## How does the computer learn?

We have a spreadsheet of **303 past patients**. For each one we know all their measurements
**and** whether they really had a blockage. That last part is the answer key.

We show the computer the measurements and the answers, let it find the pattern, then give it a
brand-new patient (no answer) and let it make its best guess. This is **machine learning**: not
magic, just careful math on many examples.

## The plan, step by step

1. **Load the data** - open the spreadsheet and fix the missing values.
2. **Look at the data (EDA)** - make charts to understand what we have.
3. **Prepare the data** - turn real-world values into numbers the model likes.
4. **Train the model** - let the computer learn the pattern, and calibrate its probabilities.
5. **Pick a cutoff** - decide when to say "high risk."
6. **Check the results** - test it on patients it has never seen.
7. **Save the model** - write a file so the dashboard can predict with no Python.
8. **Sanity check** - confirm Python and the browser give the same answer.

We use exactly **one** simple model. No comparing ten algorithms, no fancy tricks. Simple is
easier to explain and easier to trust.

---

# Step 1 - Load the data

## What is a CSV file?

A **CSV** file is plain text that looks like a spreadsheet written with commas. Each line is one
row (one patient), and commas separate the columns (the measurements).

Our file `data/raw/cleveland.csv` has **303 lines**, one per patient, and **14 columns**.

## Why do we need to fix it?

Two annoying problems:

1. **No column names.** The file starts with numbers and never says "this column is age."
   So we write the names ourselves.
2. **Missing values are written as `?`.** In real medical data some measurements were simply
   not recorded. The file uses `?` for "we don't know." A computer would think `?` is a word,
   not a missing number, and the math would break.

We fix both problems in one line using `pd.read_csv`.

## What the pieces mean

- `header=None` -> "this file has no title row, don't treat line 1 as names."
- `names=COLS` -> "use this list I made as the column names."
- `na_values="?"` -> "whenever you see `?`, treat it as **missing** (blank), not text."

## The answer column

The last column, `target`, is a severity score from **0 to 4**. We want a simple yes/no:

- score **0** -> `0` (no significant blockage)
- score **1, 2, 3, or 4** -> `1` (there is a blockage)

The line `(raw["target"] > 0).astype(int)` asks "is the score bigger than zero?" (true/false)
and turns true into `1`, false into `0`.

## What we end up with

A table named `raw` with 303 rows and 14 named columns, where every `?` is now a real blank.

In [ ]:
# ---- Import the tools we need. Each is a free library with one job. ----
import json                          # read/write the .json model file
import numpy as np                   # fast number crunching
import pandas as pd                  # tables (like Excel in Python)
import matplotlib.pyplot as plt      # basic charts
import seaborn as sns                # prettier charts

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, classification_report,
    confusion_matrix, precision_recall_curve, roc_auc_score, roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid")

RANDOM_STATE = 42         # a seed so results are identical every run
TEST_SIZE = 0.2           # keep 20% of patients aside for the final exam
SENSITIVITY_FLOOR = 0.90  # we must catch at least 90% of sick people

COLS = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
        "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"]

DATA_PATH = "data/raw/cleveland.csv"

raw = pd.read_csv(DATA_PATH, header=None, names=COLS, na_values="?")
raw["target"] = (raw["target"] > 0).astype(int)
raw.head()

---

# Step 2 - Look at the data (EDA)

## Why look before building?

**EDA** stands for **Exploratory Data Analysis**: before building a model, spend time just
*looking* at your data. It is like checking your fridge before you start cooking.

If you skip this, you can build a model on garbage and never notice. EDA catches surprises early.

## The questions we want to answer

1. How many patients, and what type is each column (number? category? yes/no)?
2. Which columns have missing values, and how many?
3. What do "normal" numbers look like (min, max, average)?
4. How many patients are sick vs healthy?
5. Do measurements actually differ between sick and healthy people?
6. Which measurements move together?

## A few terms you'll see

- **Missing value:** a blank cell. We write it `NaN` ("Not a Number").
- **Distribution:** how values are spread out.
- **Class balance:** the split between our two groups (sick vs healthy).
- **Correlation:** a number from -1 to +1 for how strongly two columns move together.

### 2a - What kind of data do we have?

`.info()` prints, for each column: how many values are not missing, the data type, and memory
use. If a column shows fewer than 303 values, it has missing data.

In [ ]:
raw.info()

### 2b - Which columns have missing data?

This counts blanks per column, keeps only the columns that have blanks, and sorts biggest first.
If a column were mostly blank we could not trust it. Ours only have a few gaps.

In [ ]:
missing = raw.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing

### 2c - The shape of each column

`.describe()` summarizes every numeric column:

- **count** - non-missing values
- **mean** - the average
- **std** - standard deviation (how spread out values are)
- **min / 25% / 50% / 75% / max** - smallest, quartiles, largest

The `50%` value is the **median**: the middle number when sorted. Half are below, half above.

In [ ]:
raw.describe().T

### 2d - Is it a fair split? (class balance)

We want to know how many patients are sick vs healthy.

- Roughly **50/50** is great - plenty of examples of both.
- **99% healthy** would be bad: a lazy model that always said "healthy" would score 99%
  and still be useless.

The left chart counts each group. The right chart shows the same split as a percentage.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

sns.countplot(x=raw["target"], ax=ax[0])
ax[0].set_title("How many patients in each group")
ax[0].set_xlabel("0 = no blockage, 1 = blockage")

ax[1].pie(raw["target"].value_counts(), labels=["no blockage", "blockage"],
          autopct="%1.1f%%")
ax[1].set_title("Share of each group")

plt.tight_layout()
plt.show()

### 2e - What do the measurements look like?

For each important numeric column we draw a **histogram**: bars where the x-axis is the value
and the height is how many patients fell in that range.

We color by class. If the two colors sit in different places, the column helps tell the groups
apart. If they overlap completely, the column is not very helpful.

In [ ]:
num_cols = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]

fig, axes = plt.subplots(2, 3, figsize=(13, 7))

for ax, col in zip(axes.ravel(), num_cols):
    sns.histplot(data=raw, x=col, hue="target", kde=True, ax=ax)
    ax.set_title(col)

plt.tight_layout()
plt.show()

### 2f - Which measurements move together?

A **correlation matrix** is a table where each cell shows how strongly two columns are related:

- `+1` (dark red) -> they rise and fall together.
- `-1` (dark blue) -> one goes up, the other goes down.
- `0` (white) -> no relationship.

This spots columns that carry the same information. Our two new features are added later in
Step 3, so this heatmap shows only the original columns.

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(raw.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("How strongly each pair of columns is related")
plt.show()

### 2g - Sick vs healthy, side by side

A **box plot** shows, for one column, the spread of values in each group. The box is the middle
50%, the line inside is the median, and dots are unusual patients (outliers).

If the boxes sit at different heights, the column separates the groups well.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))

for ax, col in zip(axes.ravel(), num_cols):
    sns.boxplot(data=raw, x="target", y=col, ax=ax)
    ax.set_title(f"{col}: healthy (0) vs sick (1)")

plt.tight_layout()
plt.show()

---

# Step 3 - Prepare the data (preprocessing)

## Why can't we feed the raw table straight to the model?

Three reasons:

1. **Missing values.** The model cannot do math with blanks. We must fill them somehow.
2. **Different scales.** Cholesterol is around 200 while `oldpeak` is around 1. Without fixing
   this, big-number columns drown out small-number ones.
3. **Categories are not numbers.** "Chest pain type 3" is not three times type 1 - they are just
   labels.

## 3a - Two new features we create

**Rate-Pressure Product (`rpp`)** = `trestbps * thalach / 100`.
Blood pressure times peak heart rate - a rough estimate of how hard the heart is working.

**Heart Rate Reserve (`hr_reserve`)** = `thalach / (220 - age)`.
`220 - age` is a common rule-of-thumb for the maximum heart rate at that age. Dividing the
patient's peak rate by that tells us how close they got to their expected maximum.

Bad math can produce `inf` (infinity), which is not a usable number. We turn those back into
"missing" so the fill-in step can handle them.

## 3b - Three kinds of columns, three recipes

**Numeric** (real measured numbers): age, blood pressure, cholesterol, peak heart rate,
ST depression, number of vessels, plus our two new features.

- Fill missing -> **median** (middle value). Median resists extreme patients better than average.
- Rescale -> **standardize**: subtract the average, divide by the spread. Every numeric column
  then centers near 0 with a similar range. No column bullies the others.

**Categorical** (labels): chest pain type, resting ECG, slope, thal.

- Fill missing -> **most common value** (mode).
- Encode -> **one-hot**: one label column becomes several yes/no columns. `cp` = 1,2,3,4 becomes
  `cp_1`, `cp_2`, `cp_3`, `cp_4`, exactly one of which is 1 per patient. This says "each type is
  its own thing," not "type 4 > type 1."

**Binary** (already yes/no): sex, fasting blood sugar, exercise angina.

- Fill missing -> most common value.
- Rescale -> standardize, like the numeric ones.

## 3c - One golden rule: never look at the test data

We learn the medians, modes, averages, and spreads **from the training patients only**, then
apply those same numbers to the test patients. Peeking at the test set while preparing is a form
of cheating: it makes our final score look better than it truly is.

## What we end up with

A reusable recipe that turns **any** raw patient into a fixed list of **24 numbers** the model
understands.

### 3d - The code for the two new features

We wrap our feature math in a small reusable object with two methods:

- `fit(X, y)` - called during training. Does nothing here, because these features learn nothing.
- `transform(X)` - the real work: copy the table, add the two columns, replace infinities.

`BaseEstimator` and `TransformerMixin` are standard scikit-learn blocks that let our class plug
into a Pipeline like any built-in step.

In [ ]:
class ClinicalFeatureEngineer(BaseEstimator, TransformerMixin):
    """Adds two calculated clinical features to the table."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["trestbps"] = pd.to_numeric(X["trestbps"], errors="coerce")
        X["thalach"] = pd.to_numeric(X["thalach"], errors="coerce")
        X["age"] = pd.to_numeric(X["age"], errors="coerce")
        X["rpp"] = (X["trestbps"] * X["thalach"]) / 100.0
        X["hr_reserve"] = X["thalach"] / (220 - X["age"])
        return X.replace([np.inf, -np.inf], np.nan)

### 3e - The full preprocessing recipe

A **Pipeline** runs steps in order. A **ColumnTransformer** sends different columns down
different pipelines. Our recipe:

1. First add the two calculated features.
2. Numeric columns -> fill-in -> rescale.
3. Categorical columns -> fill-in -> one-hot.
4. Binary columns -> fill-in -> rescale.
5. Glue the results into one wide table of 24 numbers.

We build the recipe now, but fit it in Step 4, after the training data is separated out.

In [ ]:
NUMERIC = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]
ENGINEERED = ["rpp", "hr_reserve"]
CATEGORICAL = ["cp", "restecg", "slope", "thal"]
BINARY = ["sex", "fbs", "exang"]


def build_preprocessor():
    """Different columns, different treatment."""
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    bin_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("scaler", StandardScaler()),
    ])
    return ColumnTransformer([
        ("num", num_pipe, NUMERIC + ENGINEERED),
        ("cat", cat_pipe, CATEGORICAL),
        ("bin", bin_pipe, BINARY),
    ], remainder="drop")


def build_transform_pipeline():
    """The complete recipe: first add features, then preprocess."""
    return Pipeline([
        ("engineer", ClinicalFeatureEngineer()),
        ("preprocessor", build_preprocessor()),
    ])

---

# Step 4 - Train the model

## What does "training" mean?

Show the computer many patients **whose answers we already know**, and let it adjust its
internal numbers until its guesses match the truth as often as possible.

## The one model: logistic regression

Despite the scary name it is simple:

1. Give every input feature a **weight** (a number saying how important it is).
2. Multiply each feature by its weight and add them all up. That total is the **score**.
3. A positive score pushes toward "sick"; negative pushes toward "healthy."
4. Squash the score through an **S-curve** (the sigmoid) so the answer is a probability from 0 to 1.

Training just means finding the weights that make the best guesses on the training data.

## Two settings worth knowing

- **`C=1.0`** - controls strictness. The penalty keeps weights from getting huge. Huge weights
  mean memorizing the training data instead of learning a general pattern.
- **`max_iter=2000`** - the maximum number of tries the solver gets. High, so it definitely
  finishes.

## Splitting the data: the final exam

We set aside **20% of patients** as a **test set** and never train on them. Later we grade the
model only on these unseen patients, so the score is an honest preview. We keep the same
sick/healthy ratio in both parts (a **stratified** split) and fix the seed so the split is the
same every run.

## Calibration: making probabilities honest

A raw logistic score is not always a trustworthy probability. If the model says "70% risk," we
want about 70 of 100 such patients to really be sick. Often they are not, and it is overconfident.

**Platt scaling** (sigmoid calibration) fixes this with one more small S-curve that remaps the
raw score into a better probability.

**The clever part:** we fit that curve using **out-of-fold** scores. We split the training data
into 5 parts, train on 4, score the held-out part, rotate so every part is scored by a model that
never saw it, then fit the curve on those honest scores. Otherwise the model would look more
confident than it deserves.

In [ ]:
X = raw.drop(columns=["target"])
y = raw["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

transform_pipeline = build_transform_pipeline()
Xt_train = transform_pipeline.fit_transform(X_train)

base = LogisticRegression(C=1.0, max_iter=2000, solver="lbfgs", random_state=RANDOM_STATE)
base.fit(Xt_train, y_train)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_scores = cross_val_predict(base, Xt_train, y_train, cv=skf, method="decision_function")

calibrator = LogisticRegression(C=1e6, solver="lbfgs", max_iter=1000)
calibrator.fit(oof_scores.reshape(-1, 1), y_train)

base_final = clone(base).fit(Xt_train, y_train)
Xt_test = transform_pipeline.transform(X_test)


def calibrated_proba(Xt):
    """Raw score -> calibration curve -> honest probability."""
    scores = base_final.decision_function(Xt).reshape(-1, 1)
    return calibrator.predict_proba(scores)[:, 1]


proba_test = calibrated_proba(Xt_test)
print("test ROC-AUC: %.4f" % roc_auc_score(y_test, proba_test))
print("test PR-AUC : %.4f" % average_precision_score(y_test, proba_test))

---

# Step 5 - Pick the cutoff (threshold tuning)

## What is a threshold?

The model gives every patient a probability, like 0.62. But the doctor needs a simple answer:
**high risk or not?** A **threshold** is the dividing line. If probability >= threshold we say
"high risk," otherwise "low risk." Most software defaults to 0.5, but 0.5 is wrong here.

## Why 0.5 is wrong

Two possible mistakes:

- **False negative:** we say "low risk" but the patient is actually sick. Dangerous - they miss
  the follow-up test they need.
- **False positive:** we say "high risk" but the patient is fine. Extra worry and maybe an
  unnecessary test, but nobody is hurt.

In medicine a missed disease is much worse than a false alarm. So we deliberately lower the
threshold, catching more sick people at the cost of more false alarms. A trade we happily make.

## The exact rule we follow

1. Try many thresholds from 0.050 to 1.000.
2. Keep only thresholds where **sensitivity >= 90%** (we catch at least 9 of 10 sick people).
3. Among those, pick the one with the **highest specificity** (fewest false alarms while still
   honoring the 90% promise).

## Two words, in plain English

- **Sensitivity** = "of the sick people, how many did we catch?" -> want high.
- **Specificity** = "of the healthy people, how many did we correctly reassure?" -> want high too,
  but never at the cost of sensitivity.

## A note on honesty

We choose the threshold using scores on the **training** set, because the test set is our final
exam and we must not peek at it while making decisions.

In [ ]:
def sens_spec(y_true, y_pred):
    """Return sensitivity and specificity for a set of 0/1 predictions."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    return sens, spec


proba_train = calibrated_proba(Xt_train)

best_threshold = None
best_specificity = -1.0

for t in np.arange(0.050, 1.0001, 0.001):
    decisions = (proba_train >= t).astype(int)
    sens, spec = sens_spec(y_train, decisions)
    if sens < SENSITIVITY_FLOOR:
        continue
    if spec > best_specificity:
        best_specificity = spec
        best_threshold = float(round(t, 3))

threshold = best_threshold
print("chosen threshold: %.3f" % threshold)
ts, tv = sens_spec(y_train, (proba_train >= threshold).astype(int))
print("training sensitivity: %.3f  specificity: %.3f" % (ts, tv))

---

# Step 6 - Check the results (evaluation)

## The final exam

Now we use the **test set** - the 20% the model has never seen. Everything before used the
training data; this is the first time we grade on unseen patients.

## The numbers we report

- **ROC-AUC:** how well the model ranks patients from lowest to highest risk across all
  thresholds. `0.5` = random guessing, `1.0` = perfect ordering. Above `0.8` is strong.
- **PR-AUC:** like ROC-AUC but focused on the sick class. Useful when one class is rarer.
- **Sensitivity:** of the truly sick, how many we caught.
- **Specificity:** of the truly healthy, how many we correctly left alone.
- **Brier score:** average squared error of the probabilities. Lower is better; it rewards
  honest, well-calibrated numbers, not just correct yes/no.

## Three pictures

- **ROC curve:** sensitivity against false-alarm rate. The closer to the top-left corner, the
  better. The diagonal is random guessing.
- **Precision-Recall curve:** precision (when we say sick, how often right) against recall
  (how many sick we found). Good for imbalanced data.
- **Calibration curve:** predicted probability vs what actually happened. The dashed diagonal
  is perfect honesty; points near it mean the probabilities can be trusted.

## The confusion matrix

A 2x2 table of the four outcomes on the test set:

- **True negative:** healthy, we said low risk. Good.
- **True positive:** sick, we said high risk. Good.
- **False positive:** healthy, we said high risk. A false alarm.
- **False negative:** sick, we said low risk. The dangerous miss.

In [ ]:
pred_test = (proba_test >= threshold).astype(int)

roc_auc = roc_auc_score(y_test, proba_test)
pr_auc = average_precision_score(y_test, proba_test)

fig, ax = plt.subplots(1, 3, figsize=(15, 4.4))

fpr, tpr, _ = roc_curve(y_test, proba_test)
ax[0].plot(fpr, tpr, color="#1f4e79", lw=2, label="AUC = %.3f" % roc_auc)
ax[0].plot([0, 1], [0, 1], "k--", lw=1)
ax[0].set_title("ROC curve")
ax[0].set_xlabel("false-alarm rate")
ax[0].set_ylabel("sensitivity")
ax[0].legend(loc="lower right")

prec, rec, _ = precision_recall_curve(y_test, proba_test)
ax[1].plot(rec, prec, color="#0e7490", lw=2, label="AP = %.3f" % pr_auc)
ax[1].axhline(y_test.mean(), color="k", ls="--", lw=1)
ax[1].set_title("Precision-Recall")
ax[1].set_xlabel("recall (sensitivity)")
ax[1].set_ylabel("precision")
ax[1].legend(loc="lower left")

frac_pos, mean_pred = calibration_curve(y_test, proba_test, n_bins=8, strategy="quantile")
ax[2].plot(mean_pred, frac_pos, "o-", color="#1f4e79", lw=2, label="calibrated")
ax[2].plot([0, 1], [0, 1], "k--", lw=1)
ax[2].set_title("Calibration")
ax[2].set_xlabel("predicted probability")
ax[2].set_ylabel("observed frequency")
ax[2].legend(loc="upper left")

plt.tight_layout()
plt.show()

### 6b - The four outcomes and the full report

The confusion matrix counts the four outcomes. `classification_report` adds precision, recall,
and F1 for each class, plus overall accuracy.

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, pred_test, labels=[0, 1]).ravel()
print("true negatives :", tn)
print("false positives:", fp)
print("false negatives:", fn)
print("true positives :", tp)
print()
print(classification_report(y_test, pred_test, digits=3))

In [ ]:
sens_test, spec_test = sens_spec(y_test, pred_test)

metrics = {
    "model": "logistic_regression",
    "threshold": float(threshold),
    "n_test": int(len(y_test)),
    "roc_auc": float(roc_auc),
    "pr_auc": float(pr_auc),
    "sensitivity": float(sens_test),
    "specificity": float(spec_test),
    "brier_calibrated": float(brier_score_loss(y_test, proba_test)),
}
metrics

---

# Step 7 - Save the model

## Why save anything?

The notebook runs in Python. The dashboard is a plain web page that must work with no Python at
all. So we write every number the browser needs into one small JSON file: `model_export.json`.

## What goes in the file

- **`feature_names`** - the 24 columns, in order, after preprocessing.
- **`coef` / `intercept`** - the logistic regression weights.
- **`calibrator_coef` / `calibrator_intercept`** - the Platt scaling curve.
- **`threshold`** - our chosen cutoff.
- **`num_mean` / `num_scale` / `num_median`** - how numeric columns were filled and rescaled.
- **`bin_mean` / `bin_scale` / `bin_median`** - same for yes/no columns.
- **`cat_cols` / `cat_categories`** - the categorical columns and their one-hot categories.
- **`metrics`** - the scores from Step 6, shown on the dashboard.

## One tricky detail

If a column never changes (like `fbs`), its spread is zero, and dividing by zero explodes. We
replace any zero spread with `1.0` so the math stays safe. scikit-learn does the same thing
internally, so we match it exactly.

In [ ]:
# Pull the fitted pieces back out of the pipeline.
pre = transform_pipeline.named_steps["preprocessor"]

num_imputer = pre.named_transformers_["num"].named_steps["imputer"]
num_scaler = pre.named_transformers_["num"].named_steps["scaler"]
bin_imputer = pre.named_transformers_["bin"].named_steps["imputer"]
bin_scaler = pre.named_transformers_["bin"].named_steps["scaler"]
cat_encoder = pre.named_transformers_["cat"].named_steps["encoder"]

# Numeric columns = measured numbers + the two we engineered.
num_names = NUMERIC + ENGINEERED

# Categorical feature names, as one-hot created them.
cat_names = list(cat_encoder.get_feature_names_out(CATEGORICAL))

# Binary columns keep their original names.
bin_names = BINARY

feature_names = num_names + cat_names + bin_names

# Zero spread would divide by zero; use 1.0 instead (same as scikit-learn).
num_scale = np.where(num_scaler.scale_ == 0, 1.0, num_scaler.scale_).tolist()
bin_scale = np.where(bin_scaler.scale_ == 0, 1.0, bin_scaler.scale_).tolist()

payload = {
    "feature_names": list(feature_names),
    "coef": base_final.coef_.ravel().tolist(),
    "intercept": float(base_final.intercept_[0]),
    "classes": [int(c) for c in base_final.classes_],

    "calibrator_coef": calibrator.coef_.ravel().tolist(),
    "calibrator_intercept": float(calibrator.intercept_[0]),

    "threshold": float(threshold),

    "num_cols": num_names,
    "num_mean": num_scaler.mean_.tolist(),
    "num_scale": num_scale,
    "num_median": num_imputer.statistics_.tolist(),

    "bin_cols": bin_names,
    "bin_mean": bin_scaler.mean_.tolist(),
    "bin_scale": bin_scale,
    "bin_median": bin_imputer.statistics_.tolist(),

    "cat_cols": CATEGORICAL,
    "cat_categories": [list(map(float, cats)) for cats in cat_encoder.categories_],

    "metrics": metrics,
}

with open("model_export.json", "w") as f:
    json.dump(payload, f, indent=2)

print("saved model_export.json with", len(payload["feature_names"]), "features")

---

# Step 8 - Sanity check: does Python agree with the browser?

## What are we checking?

The dashboard is a web page that must run with **no Python at all**. It does the same math in
JavaScript. If the two ever disagree, the number a doctor reads on screen would be wrong. So the
last thing we do is prove they match.

## The test patient

This is the same patient the dashboard loads by default, so you can reproduce it by eye.

A 58-year-old man with no typical chest pain, moderately high blood pressure and cholesterol, a
peak heart rate of 118, exercise-induced chest pain, and two major vessels flagged. Lots of
risk factors, so we expect a high number.

## The two paths

1. **The Python path** - push the patient through the real fitted pipeline. This is ground truth.
2. **The browser path** - ignore the fitted objects entirely and rebuild the answer using only
   the numbers written in `model_export.json`: the saved averages, spreads, medians, one-hot
   category lists, weights, calibration curve, and threshold. This is exactly what the JavaScript
   in `index.html` does.

If the dashboard ever drifts from the notebook, these two will stop matching.

## What good looks like

The difference should be `0.00e+00` - not "small", but exactly zero. Any nonzero gap means the
export and the browser have drifted apart and someone needs to re-export.

In [ ]:
# The same patient the dashboard loads by default.
patient = {"age": 58, "sex": 1, "cp": 4, "trestbps": 158, "chol": 285, "fbs": 0,
           "restecg": 2, "thalach": 118, "exang": 1, "oldpeak": 2.6, "slope": 2,
           "ca": 2, "thal": 7}


def browser_predict(s, p):
    """Recompute the answer using ONLY the exported JSON - no fitted objects.

    This mirrors the JavaScript in index.html line for line.
    """
    raw_vec = dict(s)
    raw_vec["rpp"] = s["trestbps"] * s["thalach"] / 100.0
    raw_vec["hr_reserve"] = s["thalach"] / (220 - s["age"])

    for col in p["cat_cols"]:
        for cat in p["cat_categories"][p["cat_cols"].index(col)]:
            raw_vec[f"{col}_{cat}"] = 1.0 if s[col] == cat else 0.0

    z = p["intercept"]
    for i, name in enumerate(p["feature_names"]):
        x = raw_vec.get(name, 0.0)
        if name in p["num_cols"]:
            j = p["num_cols"].index(name)
            if pd.isna(x):
                x = p["num_median"][j]
            x = (x - p["num_mean"][j]) / p["num_scale"][j]
        elif name in p["bin_cols"]:
            j = p["bin_cols"].index(name)
            x = (x - p["bin_mean"][j]) / p["bin_scale"][j]
        z += p["coef"][i] * x

    cz = p["calibrator_coef"][0] * z + p["calibrator_intercept"]
    return 1 / (1 + np.exp(-cz))


# Path 1: the real fitted pipeline (ground truth).
prob_python = float(calibrated_proba(transform_pipeline.transform(pd.DataFrame([patient])))[0])

# Path 2: the exported numbers only (what the browser computes).
prob_browser = float(browser_predict(patient, payload))

print("python  path: %.6f" % prob_python)
print("browser path: %.6f" % prob_browser)
print("difference  : %.2e" % abs(prob_python - prob_browser))
print("threshold   : %.3f" % threshold)
print("verdict     :", "HIGH" if prob_browser >= threshold else "LOW")

## Reading the result

Both paths print the same number and the difference is exactly zero, so the dashboard and the
notebook are in perfect agreement. The verdict comes from comparing that number to our threshold.

## One last word of caution

A probability like 0.996 does not mean certainty. It means that among patients who looked like
this one, about 99.6% had a blockage. It is a statement about a group, not a promise about one
person. This model is a **research prototype** and is not approved for clinical use.